In [1]:
try:
    import dolfinx
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh" -O "/tmp/fenicsx-install.sh" && bash "/tmp/fenicsx-install.sh"
    import dolfinx

In [2]:
from mpi4py import MPI
from dolfinx import mesh
domain = mesh.create_unit_square(MPI.COMM_WORLD, 8, 8, mesh.CellType.quadrilateral)

In [3]:
from dolfinx.fem import functionspace
V = functionspace(domain, ("Lagrange", 1))

In [4]:
from dolfinx import fem
uD = fem.Function(V)
uD.interpolate(lambda x: 1 + x[0]**2 + 2 * x[1]**2)

In [5]:
import numpy
# Create facet to cell connectivity required to determine boundary facets
tdim = domain.topology.dim
fdim = tdim - 1
domain.topology.create_connectivity(fdim, tdim)
boundary_facets = mesh.exterior_facet_indices(domain.topology)

In [6]:
boundary_dofs = fem.locate_dofs_topological(V, fdim, boundary_facets)
bc = fem.dirichletbc(uD, boundary_dofs)

In [7]:
import ufl
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

In [8]:
from dolfinx import default_scalar_type
f = fem.Constant(domain, default_scalar_type(-6))

In [9]:
a = ufl.dot(ufl.grad(u), ufl.grad(v)) * ufl.dx
L = f * v * ufl.dx

In [11]:
from dolfinx.fem.petsc import LinearProblem
problem = LinearProblem(a, L, bcs=[bc], petsc_options={"ksp_type": "preonly", "pc_type": "lu"}, \
                        petsc_options_prefix="linear_problem_")
uh = problem.solve()

In [12]:
V2 = fem.functionspace(domain, ("Lagrange", 2))
uex = fem.Function(V2)
uex.interpolate(lambda x: 1 + x[0]**2 + 2 * x[1]**2)

In [13]:
L2_error = fem.form(ufl.inner(uh - uex, uh - uex) * ufl.dx)
error_local = fem.assemble_scalar(L2_error)
error_L2 = numpy.sqrt(domain.comm.allreduce(error_local, op=MPI.SUM))

In [14]:
error_max = numpy.max(numpy.abs(uD.x.array-uh.x.array))
# Only print the error on one process
if domain.comm.rank == 0:
    print(f"Error_L2 : {error_L2:.2e}")
    print(f"Error_max : {error_max:.2e}")

Error_L2 : 8.24e-03
Error_max : 3.55e-15


In [15]:
import pyvista
print(pyvista.global_theme.jupyter_backend)

None


In [17]:
from dolfinx import plot
pyvista.OFF_SCREEN = True
domain.topology.create_connectivity(tdim, tdim)
topology, cell_types, geometry = plot.vtk_mesh(domain, tdim)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

In [18]:
plotter = pyvista.Plotter(off_screen=True)
plotter.add_mesh(grid, show_edges=True)
plotter.view_xy()

import os
# This is to know where you are situated in yourr google drive
print(os.getcwd())

figure = plotter.screenshot("mesh.png")

/home/dmsm/a.brugnoli/GitHub/fea-class-3A


In [19]:
u_topology, u_cell_types, u_geometry = plot.vtk_mesh(V)

In [20]:
u_grid = pyvista.UnstructuredGrid(u_topology, u_cell_types, u_geometry)
u_grid.point_data["u"] = uh.x.array.real
u_grid.set_active_scalars("u")
u_plotter = pyvista.Plotter(off_screen=True)
u_plotter.add_mesh(u_grid, show_edges=True)
u_plotter.view_xy()

figure = u_plotter.screenshot("solution_2d.png")

In [21]:
warped = u_grid.warp_by_scalar()
plotter2 = pyvista.Plotter(off_screen=True)
plotter2.add_mesh(warped, show_edges=True, show_scalar_bar=True)

figure = plotter2.screenshot("solution_3d.png")